if k == experts its called Dense MoE </br>
if k < experts its called spares MoE </br> </br>
torch.nn.ModuleList is a PyTorch container class used to hold submodules in a list. <br/>
<b>Key Features</b> <br/> <br/>
<b>1. Parameter Registration:</b> Modules stored in a nn.ModuleList are automatically registered as parameters of the parent nn.Module. This allows the optimizer to find and update their weights during training. <br/><br/>
<b>2. Iterable and Indexable:</b> nn.ModuleList can be indexed like a regular Python list and iterated over in the forward method of a custom nn.Module subclass.<br/><br/>
<b>3. Custom Forward Pass:</b> Unlike nn.Sequential, nn.ModuleList does not implement a forward method automatically. This design allows for more flexibility in the data flow, such as residual connections or non-sequential architectures, as the user defines the exact computation in the custom forward method. 

In [ ]:

import torch
import torch.nn as nn
class FeedForward(nn.Module):
      def __init__(self, in_features, out_features):
            super().__init__()
            self.W = nn.Parameter(torch.randn(out_features, in_features))
            self.b = nn.Parameter(torch.randn(out_features))

      def forward(self, x):
            return self.x+self.W + self.b

class MoELayer(nn.Module):
        def __init__(self, d_model, d_ff, num_experts, k=2):
            super().__init__()
            self.num_experts = num_experts
            self.k = k


            # implement experts = dependent FFN
            self.expert = nn.ModuleList(
                  [ FeedForward(d_model, d_ff) for _ in range (num_experts) ]
            )
            
            # Router / Gating network 
            self.router = nn.Linear(d_model, num_experts) # classification which expert to pick  - select the very light weighted network 

        def forward(self, x):
              """
              x shape : [batch_size, seq_len, d_model num_experts] 
              # extract the dimensions 
              """
              
              # 1. Router Logits
              # Flattem the input matrix
              B, T, D = x.shape
              x_flatten = x.view(B*T, D) # treat each token independently
               
             # 1. Router Logits
             logits = self.router(x_flatten)

            # 2. Convert Logits into Probabilities 
            probabilities = torch.softmax(logits, dim=-1)
        
            # Select the top_K experts  per token
            topk_vals, topk_idx =  torch.topk(probabilities, self.k, dim=-1)

            # 3. Normalize top-k probabilities
            topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)
           
            # 4. Dispatch tokens to experts 
            output =  torch.zeros_like(x_flatten)
            
            for expert_id in range(self.num_experts):
              # Find tokens assigned to this expert
              mask = (topk_idx == expert_id) # (tokens, K)
              if not mask.any():
                   continue
              # Get token indices and corresponding weights
              token_indices, k_indices = mask.nonzero(as_tuple = True)

              # Gather the actual token vectors for this expert
              expert_input = x_flatten[token_indices]  # (num_assigned, D)

              # Run them through this expert's FFN
              expert_output = self.expert[expert_id](expert_input)  # (num_assigned, d_ff) 

              # Get the gating weights for each assigned token (combine the expert_output)
              weights = topk_vals[token_indices, k_indices].unsqueeze(-1)  # (num_assigned, 1) 
             

              # Weighted accumulation into output
              # BUG NOTE: d_ff must equal D (d_model) for this addition to work,
              # or you need a projection layer. Assuming d_ff == d_model here.
              # output.index_add_(0, token_indices, weights * expert_output)

              output[token_indices] += expert_output * weights

            # 5. Reshape output back to [B, T, D]
            output = output.view(B, T, D)

        return output


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x107ad3c70>>
Traceback (most recent call last):
  File "/Users/abroadhub/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
